In [8]:
import pandas as pd

In [9]:
air_quality = pd.read_csv("hourly_88101_2024.csv", low_memory=False)

# basic structure and inferred data types
print(f"Shape: {air_quality.shape}\n")
air_quality.info()

# summary of each column
column_summary = pd.DataFrame({
    "dtype": air_quality.dtypes.astype(str),
    "missing": air_quality.isna().sum(),
    "unique_values": air_quality.nunique(dropna=False),
    "suggested_type": [
        "category" if dtype == "object" and unique / len(air_quality) < 0.5 # if the unique-total ratio is less than 50%, consider it a category
        else str(dtype)
        for dtype, unique in zip(air_quality.dtypes, air_quality.nunique())
    ]
})

display(column_summary)

# frequency tables for each column
for column in air_quality.columns:
    print(f"\n--- {column} ---")
    display(air_quality[column].value_counts(dropna=False).head(20))

# preview the data
display(air_quality.head())

ParserError: Error tokenizing data. C error: out of memory

* Can we use a combination of keys for our primary key?

* Since this is hourly data, we might have unique values by long,lat,date/time

In [ ]:
# Identify rows with duplicate longitude, latitude, and hour values
key_columns = ["Longitude", "Latitude", "Date Local", "Time Local", "Method Code"]

missing_columns = [column for column in key_columns if column not in air_quality.columns]
if missing_columns:
    raise KeyError(f"Missing required columns: {missing_columns}")

duplicate_mask = air_quality.duplicated(subset=key_columns, keep=False)
duplicates = air_quality.loc[duplicate_mask].sort_values(key_columns)

print(f"Duplicate rows found: {len(duplicates)}")
display(duplicates[key_columns])

Duplicate rows found: 778636


,Longitude,Latitude,Date Local,Time Local,Method Code
2165299,-155.913417,19.509778,2024-01-01,00:00,209
2173917,-155.913417,19.509778,2024-01-01,00:00,209
2165300,-155.913417,19.509778,2024-01-01,01:00,209
2173918,-155.913417,19.509778,2024-01-01,01:00,209
2165302,-155.913417,19.509778,2024-01-01,03:00,209
...,...,...,...,...,...
3468700,-71.056061,42.325186,2024-12-31,21:00,636
3460113,-71.056061,42.325186,2024-12-31,22:00,636
3468701,-71.056061,42.325186,2024-12-31,22:00,636
3460114,-71.056061,42.325186,2024-12-31,23:00,636


In [ ]:
duplicates

,State Code,County Code,Site Num,Parameter Code,POC,Latitude,Longitude,Datum,Parameter Name,Date Local,...,Units of Measure,MDL,Uncertainty,Qualifier,Method Type,Method Code,Method Name,State Name,County Name,Date of Last Change
2242416,15,3,10,88101,1,21.323745,-158.088613,WGS84,PM2.5 - Local Conditions,2024-01-01,...,Micrograms/cubic meter (LC),0.1,NaN,NaN,FEM,238,Teledyne T640X at 16.67 LPM - Broadband spectr...,Hawaii,Honolulu,2024-06-10
2251082,15,3,10,88101,21,21.323745,-158.088613,WGS84,PM2.5 - Local Conditions,2024-01-01,...,Micrograms/cubic meter (LC),0.1,NaN,NaN,FEM,738,Teledyne T640X at 16.67 LPM (Corrected) - Broa...,Hawaii,Honolulu,2024-06-10
2242417,15,3,10,88101,1,21.323745,-158.088613,WGS84,PM2.5 - Local Conditions,2024-01-01,...,Micrograms/cubic meter (LC),0.1,NaN,NaN,FEM,238,Teledyne T640X at 16.67 LPM - Broadband spectr...,Hawaii,Honolulu,2024-06-10
2251083,15,3,10,88101,21,21.323745,-158.088613,WGS84,PM2.5 - Local Conditions,2024-01-01,...,Micrograms/cubic meter (LC),0.1,NaN,NaN,FEM,738,Teledyne T640X at 16.67 LPM (Corrected) - Broa...,Hawaii,Honolulu,2024-06-10
2242418,15,3,10,88101,1,21.323745,-158.088613,WGS84,PM2.5 - Local Conditions,2024-01-01,...,Micrograms/cubic meter (LC),0.1,NaN,NaN,FEM,238,Teledyne T640X at 16.67 LPM - Broadband spectr...,Hawaii,Honolulu,2024-06-10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3180788,23,3,1011,88101,4,46.682299,-68.016195,WGS84,PM2.5 - Local Conditions,2024-09-16,...,Micrograms/cubic meter (LC),5.0,NaN,NaN,FEM,170,Met One BAM-1020 Mass Monitor w/VSCC - Beta At...,Maine,Aroostook,2024-10-03
3176371,23,3,1011,88101,3,46.682299,-68.016195,WGS84,PM2.5 - Local Conditions,2024-09-16,...,Micrograms/cubic meter (LC),0.1,NaN,NaN,FEM,638,Teledyne T640X at 16.67 LPM w/Network Data Ali...,Maine,Aroostook,2024-10-03
3180789,23,3,1011,88101,4,46.682299,-68.016195,WGS84,PM2.5 - Local Conditions,2024-09-16,...,Micrograms/cubic meter (LC),5.0,NaN,NaN,FEM,170,Met One BAM-1020 Mass Monitor w/VSCC - Beta At...,Maine,Aroostook,2024-10-03
3176372,23,3,1011,88101,3,46.682299,-68.016195,WGS84,PM2.5 - Local Conditions,2024-09-16,...,Micrograms/cubic meter (LC),0.1,NaN,NaN,FEM,638,Teledyne T640X at 16.67 LPM w/Network Data Ali...,Maine,Aroostook,2024-10-03


* Apparently not. So we probably have to generate our own key since we cant find a reliable combination

In [12]:
air_quality = pd.read_csv("hourly_88101_2024.csv", low_memory=False, sep=",", dtype="string", on_bad_lines="warn")

# Candidate columns that are meaningful for partitioning
groupby_columns = [
    "State Name",
    "State Code",
    "County",
    "County Code",
    "Site Num",
    "City",
    "Parameter Name",
    "Method Code",
]

# Add date-based partition candidates when available
if "Date Local" in air_quality.columns:
    air_quality["partition_year"] = pd.to_datetime(
        air_quality["Date Local"], errors="coerce"
    ).dt.year
    air_quality["partition_month"] = pd.to_datetime(
        air_quality["Date Local"], errors="coerce"
    ).dt.to_period("M").astype(str)

groupby_columns += ["partition_year", "partition_month"]

for column in groupby_columns:
    if column not in air_quality.columns:
        continue

    group_sizes = (
        air_quality.groupby(column, dropna=False)
        .size()
        .sort_values(ascending=False)
    )
    required_columns = set(groupby_columns) | {"Date Local"}

    chunks = []

    for _, chunk in pd.read_csv(
        "hourly_88101_2024.csv",
        usecols=lambda column: column in required_columns,
        dtype="string",
        chunksize=50_000,
        engine="python",
        on_bad_lines="warn",
    ):
        chunks.append(chunk)

    if not chunks:
        raise ValueError(f"No data found while processing '{column}'")

    air_quality = pd.concat(chunks, ignore_index=True)
    print(f"\n--- Group sizes by {column} ---")
    print(f"Number of groups: {len(group_sizes)}")
    print(f"Largest group: {group_sizes.max():,}")
    print(f"Smallest group: {group_sizes.min():,}")
    print(f"Median group size: {group_sizes.median():,.0f}")
    display(group_sizes.head(20).rename("row_count").to_frame())

ParserError: Error tokenizing data. C error: out of memory